In [13]:
from google.colab import drive
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Add the folder containing data_utils.py to the system path
sys.path.append('/content/drive/MyDrive/data')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.8 MB/s eta 0:00:00


In [3]:
# Standard libraries
import os
import cv2
import sys
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Metrics and evaluation
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

# Keras Tuner
from keras_tuner import HyperParameters
from keras_tuner.tuners import RandomSearch

# Custom utility functions
from data_utils import load_sample_dataframe_from_dir, load_dataset_from_df, load_dataset_from_dir


# Data Import

In [4]:
# Use Kaggle data as base

# --- Paths ---
normal_data_path = '/content/drive/MyDrive/data/chest_xray/train/NORMAL'
pneumonia_data_path = '/content/drive/MyDrive/data/chest_xray/train/PNEUMONIA'
test_data_path = '/content/drive/MyDrive/data/chest_xray/test'

# --- Build and split DataFrames ---
train_df = pd.DataFrame(columns=['filename', 'class'])
train_df = load_sample_dataframe_from_dir(train_df, normal_data_path, 'NORMAL', sample_size=1341, seed=42)
train_df = load_sample_dataframe_from_dir(train_df, pneumonia_data_path, 'PNEUMONIA', sample_size=3875, seed=42)

# Stratified train/val split
search_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['class'], random_state=42)

# --- Create datasets ---
search_ds = load_dataset_from_df(search_df, "filename", "class", img_size=(224, 224), batch_size=32)
val_ds = load_dataset_from_df(val_df, "filename", "class", img_size=(224, 224), batch_size=32)
test_ds = load_dataset_from_dir(test_data_path, img_size=(224, 224), batch_size=32, gaussian_blur=False)


Found 4172 validated image filenames belonging to 2 classes.
Found 1044 validated image filenames belonging to 2 classes.
Found 624 images belonging to 2 classes.


## CNN Model Search
The function below defines a baseline Convolutional Neural Network (CNN) architecture, which serves as a benchmark to evaluate how different data configurations impact model performance.

This model consists of two configurable convolutional blocks followed by max-pooling layers to extract and downsample key features. A fully connected dense layer captures higher-level patterns before passing to the final classification layer.

The output layer uses a sigmoid activation to assign probabilities for Normal vs Pneumonia classification in this binary setup.

In [5]:
def build_model(hp):
    input_shape = (224, 224, 1)
    num_classes = 1

    model = models.Sequential()

    # Conv Block 1
    model.add(layers.Conv2D(
        filters=hp.Int('conv1_filters', min_value=32, max_value=64, step=16),
        kernel_size=(3, 3),
        padding='same',
        input_shape=input_shape
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Conv Block 2
    model.add(layers.Conv2D(
        filters=hp.Int('conv2_filters', min_value=64, max_value=128, step=32),
        kernel_size=(3, 3),
        padding='same'
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Flatten + Dense
    model.add(layers.Flatten())
    model.add(layers.Dense(
        units=hp.Int('dense_units', min_value=64, max_value=256, step=64),
        kernel_regularizer=regularizers.l2(0.001)
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(hp.Float('dropout_rate', 0.3, 0.5, step=0.1)))

    # Output
    model.add(layers.Dense(num_classes, activation='sigmoid'))

    # Compile
    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )

    return model


In [6]:
# Define Model Tuner

tuner = RandomSearch(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=25,
    executions_per_trial=1,
    directory='/content/tuner_results',  # local (not Drive)
    project_name='pneumonia_cnn'
)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# Optional: Stop early if no improvement
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Start tuning search
tuner.search(
    search_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping]
)


Trial 25 Complete [00h 00m 19s]

Best val_accuracy So Far: 0.982758641242981
Total elapsed time: 03h 34m 12s


In [11]:
# Display the best hyperparameters
tuner.results_summary()


Results summary
Results in /content/tuner_results/pneumonia_cnn
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 09 summary
Hyperparameters:
conv1_filters: 64
conv2_filters: 64
dense_units: 64
dropout_rate: 0.3
learning_rate: 0.0001
Score: 0.982758641242981

Trial 17 summary
Hyperparameters:
conv1_filters: 64
conv2_filters: 128
dense_units: 192
dropout_rate: 0.4
learning_rate: 0.0001
Score: 0.9818007946014404

Trial 06 summary
Hyperparameters:
conv1_filters: 48
conv2_filters: 128
dense_units: 256
dropout_rate: 0.3
learning_rate: 0.0001
Score: 0.9798850417137146

Trial 14 summary
Hyperparameters:
conv1_filters: 64
conv2_filters: 96
dense_units: 256
dropout_rate: 0.3
learning_rate: 0.0001
Score: 0.9798850417137146

Trial 18 summary
Hyperparameters:
conv1_filters: 32
conv2_filters: 64
dense_units: 64
dropout_rate: 0.3
learning_rate: 0.0001
Score: 0.9798850417137146

Trial 13 summary
Hyperparameters:
conv1_filters: 48
conv2_filters: 128
dense_units: 128
dropout

In [17]:
# Get best model from tuner
best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(1)[0]

# Define save directory
save_dir = '/content/drive/MyDrive/data/cnn_model/'
os.makedirs(save_dir, exist_ok=True)

# Save the best model (.keras format)
model_path = os.path.join(save_dir, 'best_cnn_model.keras')
best_model.save(model_path)

# Save best hyperparameters to a text file
hparams_path = os.path.join(save_dir, 'best_cnn_hyperparameters.txt')
with open(hparams_path, 'w') as f:
    for param in best_hps.values:
        f.write(f"{param}: {best_hps.get(param)}\n")

print(f"✅ Best model saved to: {model_path}")
print(f"✅ Hyperparameters saved to: {hparams_path}")

✅ Best model saved to: /content/drive/MyDrive/data/cnn_model/best_cnn_model.keras
✅ Hyperparameters saved to: /content/drive/MyDrive/data/cnn_model/best_cnn_hyperparameters.txt
